## Import files

In [1]:
# Import data processing modules
import pandas as pd # used throughout the code
import numpy as np # used throughout the code
import country_converter as coco # used for converting country names to ISO codes
from sklearn.impute import KNNImputer # used for imputation of values for missing country-climate zone combinations
#import scipy as sp # used for correlation analysis after disaggregation


# Import visualization modules
import matplotlib.pyplot as plt # used for simple bar charts and as basis for plotly and seaborn library
import plotly.express as px # used for all stacked bar charts
import plotly.graph_objects as go # used for marimekko plot and for annotating stacked bar charts
import seaborn as sns # used for scatterplot and kdeplot in data and sensitivity analysis section after disaggregation
from textwrap import fill # used for scatterplot in data analysis section after disaggregation

# Display options
pd.options.display.float_format = '{:,.2f}'.format #limits printed decimal points to two
#pd.reset_option('^display.', silent=True) #option to reset the previous display option

In [2]:
urbanity = False
climate_calc = False #set this to true to reproduce climate zone with NUTS region matching

if climate_calc == True:
    import geopandas as gpd
    import xarray as xr #used to open netcdf climate file

    from matplotlib.colors import ListedColormap
    from matplotlib import colormaps
    from matplotlib.gridspec import GridSpec
    from matplotlib import rcParams

    #%% Global plotting settings
    rcParams['font.family'] = 'Bitstream Vera Sans'
    rcParams['font.size'] = 10

#TODOS
- test how region_nuts compares to region_bld in absolute terms for region_gea
- calculate population per nuts3 and urbanity
- calculate building stock per nuts3 and urbanity
- tenure

- material intensity
- change the model to include unoccupied dwellings for material calculation

## Add Approach: add another column with NUTS3 labels to regions label file

In [4]:
#Functions

def nuts3code_to_region_nuts(input):
      input = pd.merge(input,code_to_region_nuts, how='left') #adding region labels
      for nuts3code in input[input['region_nuts'].isna()]['NUTS-3 Code']:
            if nuts3code in code_to_region_nuts['code_2021'].values: #adding region labels with name changes from 2021 to 2024
                  input.loc[input['NUTS-3 Code']==nuts3code,'region_nuts'] = code_to_region_nuts.loc[code_to_region_nuts['code_2021'] == nuts3code, 'region_nuts'].values[0]
                  input.loc[input['NUTS-3 Code']==nuts3code,'region_bld'] = code_to_region_nuts.loc[code_to_region_nuts['code_2021'] == nuts3code, 'region_bld'].values[0]
                  #not included are Extra-Regio regions (ZZZ), i.e. air and waterways, as well as Switzerland and Norway
      input.drop(['NUTS-3 Code', 'code_1999', 'code_2003', 'code_2006', 'code_2010', 'code_2013',
       'code_2016', 'code_2021'], axis=1, inplace=True)
      input.dropna(subset='region_nuts', inplace=True)
      return input

def all_rows_contained(df1, df2):
    """Check if all rows in df1 are contained in df2"""
    merged = df1.merge(df2, how='left', indicator=True)
    return (merged['_merge'] == 'both').all()

### Expanding the index

In [5]:
# Region Labels Detailed: creating labels that include an extra level for NUTS3 regions

#importing NUTS labels
if urbanity == True:
    usecols = ['Country code', 'NUTS-3 Code', 'Urban-Rural typology']
else:
    usecols = ['Country code', 'NUTS-3 Code']
nuts_lab = pd.read_excel('data/input_detailing_NUTS/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS-3 Typologies', header=0, usecols=usecols)
nuts_lab['iso3'] = coco.convert(names=nuts_lab['Country code'], to='ISO3')

#importing region labels
region_lab = pd.read_csv('data/input_csv_SSP_2023_resid/regions_R61.csv')
region_lab['iso3']=region_lab.region_bld.str[-3:]

#appending NUTS labels to region labels where available
region_nuts_lab = pd.merge(nuts_lab, region_lab, on='iso3',how='left') #'outer'
region_nuts_lab['region_nuts'] = [str(x) + '-' + str(y) for x, y in zip(region_nuts_lab['region_bld'], region_nuts_lab['NUTS-3 Code'])]
region_nuts_lab['region_nuts'] = region_nuts_lab['region_nuts'].str.strip('-nan')

#cleaning df and harmonising with input dataframes
if urbanity == True:
    region_nuts_lab = region_nuts_lab.replace({'predominantly urban':'urb', 'intermediate':'urb', 'predominantly rural':'rur'})
    region_nuts_lab = region_nuts_lab.rename({'Urban-Rural typology':'urt'}, axis=1)
    
    #adding rural options to LUX, MLT, CYP
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('LUX')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('MLT')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('CYP')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab_urb = region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1)

    region_nuts_lab.drop(['urt'], axis=1, inplace=True)
    region_nuts_lab.drop_duplicates(inplace=True)

#define index to use when importing data
code_to_region_nuts = region_nuts_lab[['NUTS-3 Code', 'region_nuts', 'region_bld']] #input to the function nuts3code_to_region_nuts()

#export
region_nuts_lab = region_nuts_lab.drop(['iso3','NUTS-3 Code', 'Country code'], axis=1)
region_nuts_lab.to_csv('data/input_csv_NUTS_2025_resid/regions_R61_nuts.csv', index=False)
region_nuts_lab.to_csv('data/input_csv_NUTS_2025_resid/regions_R61_nuts.csv', index=False)


#checks
print('Missing values: ',region_nuts_lab.isna().sum().sum())
print('Duplicates: ',region_nuts_lab.duplicated().sum().sum())
print('NUTS3 regions: ',len(region_nuts_lab.region_nuts.unique()))
print('Index still contained: ', region_lab.drop('iso3', axis=1)[region_lab.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True)\
      .equals(region_nuts_lab.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [6]:
nuts2021_2024 = pd.read_excel('data/input_detailing_NUTS/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS2021- NUTS2024', header=0)
nuts2021_2024['Code 2021'] = nuts2021_2024['Code 2021'].fillna(method='ffill')
nuts2021_2024['Code 2024'] = nuts2021_2024['Code 2024'].fillna(method='bfill')
nuts2021_2024 = nuts2021_2024.rename(columns={'Code 2021':'code_2021', 'Code 2024':'code_2024'})
nuts2021_2024 = nuts2021_2024[nuts2021_2024['NUTS level']==3]

nuts_changes = pd.read_csv('data/input_detailing_NUTS/nuts_changes.csv')
nuts_changes = nuts_changes[nuts_changes['typology']=='nuts_level_3']

nuts_changes_2024 = pd.merge(nuts_changes, nuts2021_2024, on='code_2021', how='right').sort_index(axis=1).loc[:,'code_1999':'code_2024']
nuts_changes_2024 = nuts_changes_2024.fillna(method='ffill')
nuts_changes_2024 = nuts_changes_2024.rename(columns={'code_2024':'NUTS-3 Code'})
code_to_region_nuts = pd.merge(code_to_region_nuts, nuts_changes_2024, on='NUTS-3 Code', how='left')
code_to_region_nuts.to_csv('data/input_detailing_NUTS/code_to_region_nuts.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/320109285.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts2021_2024['Code 2021'] = nuts2021_2024['Code 2021'].fillna(method='ffill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/320109285.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts2021_2024['Code 2024'] = nuts2021_2024['Code 2024'].fillna(method='bfill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/320109285.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nuts_changes_2024 = nuts_changes_2024.fillna(method='ffill')


In [7]:
#Climate Zones: overlaying NUTS regions with matching climate zones

if climate_calc == True:
       #import climate zones file
       ds = xr.open_dataset('data/input_detailing_NUTS/climate_zones.nc')
       df = ds.to_dataframe()

       #assigning literal climate zone names to match with the clim variable
       df = df.combined
       climate_zones_names = pd.read_csv('data/input_detailing_NUTS/climate_zones_names.csv', header=None)
       climate_zones_names.columns = ['combined', 'clim']
       climate_zones_names = climate_zones_names.iloc[1:]
       replacement_map = pd.Series(climate_zones_names.clim.values, index=climate_zones_names.combined).to_dict()
       df = df.replace(replacement_map)
       df.to_csv('data/input_detailing_NUTS/climate_zones.csv')

       # merge climate zones with NUTS regions, assigning each NUTS region the climate zone that is most common to the area, in case of equal split, one of the zones is selected randomly
       climate = pd.read_csv('data/input_detailing_NUTS/climate_zones.csv')
       gdf = gpd.GeoDataFrame(climate,geometry=gpd.points_from_xy(climate.lon,climate.lat)).drop(['lat', 'lon'], axis=1)
       nuts = gpd.read_file('data/input_detailing_NUTS/NUTS_RG_20M_2024_3035.gpkg') #need to match the coordinate systems between gdf and nuts
       nuts = nuts.loc[nuts['NUTS_ID'].str.len() == 5] #only NUTS3 level
       gdf = gdf.set_crs(epsg=4326)
       gdf = gdf.to_crs(epsg=3035)
       nuts_climate = nuts.sjoin_nearest(gdf, how='left')
       nuts_climate = nuts_climate.drop(['index_right', 'geometry', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME',
              'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE'], axis=1).groupby(by=['NUTS_ID']).agg(lambda x: pd.Series.mode(x)[0]).reset_index()
       nuts_climate.columns = ['NUTS-3 Code', 'clim']
       nuts_climate.to_csv('data/input_detailing_NUTS/climate_nuts.csv', index=False)

input = pd.read_csv('data/input_detailing_NUTS/climate_nuts.csv', index_col=[0])

input = nuts3code_to_region_nuts(input)

#import file
filename= 'climatic_zones_rev'
input_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

#harmonize index
input_nuts = pd.merge(input_original.drop('clim', axis=1).drop_duplicates(), input, on=['region_bld'], how='right')

#export
if input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT']).sum()>0:
       input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
elif input_nuts.duplicated().any():
       input_nuts.drop_duplicates(inplace=True)
       print('dropped duplicates')
else:
       print('no double entries')
input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#index
region_nuts_clim = input_nuts.copy()

#checks
print('Climate zones missing from EU-NUTS dataset:', set(input_original.clim.unique()) - set(input_nuts.clim.unique()))
print('Regions in which missing climate zones occur:', input_original[input_original['clim'].isin(set(input_original.clim.unique()) - set(input_nuts.clim.unique()))].region_bld.unique())

#checks
print('Missing values: ',input_nuts.isna().sum().sum())
print('Duplicates: ',input_nuts.duplicated().sum().sum())
print('NUTS3 regions: ',len(input_nuts.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(input_original[input_original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), input_nuts.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

dropped duplicates
Climate zones missing from EU-NUTS dataset: {'Zone_5B Cool Dry', 'Zone_7B Very cold Dry', 'Zone_0B Extremely hot Dry', 'Zone_6B Cold Dry', 'Zone_4B Mixed Dry', 'Zone_1B Very hot Dry'}
Regions in which missing climate zones occur: ['R32BRA' 'R32CAS-CAU' 'R32CAS-OTH' 'R32CHN' 'R32IND' 'R32MEA-H'
 'R32MEA-M' 'R32MEX' 'R32NAF' 'R32OAS-L-PAS' 'R32PAK' 'R32SSA-L'
 'R32SSA-M']
Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [39]:
    #Check which country-climatezone combinations only appear when having NUTS detail --> important to always use detailed clim file
    mapped = input_nuts.copy()

    # OLD CODE ONLY FOR COMPARISON! #TODO: remove this

    #import file
    filename= 'climatic_zones_rev'
    input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

    #merge files
    input_nuts = pd.merge(input, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), on=['region_bld'], how='right')

    duplic = input_nuts.copy()

    # Finding combinations that only appear in one of two dataframes

    # Get unique combinations for each dataframe
    duplic_combinations = set(zip(duplic['region_bld'], duplic['clim']))
    mapped_combinations = set(zip(mapped['region_bld'], mapped['clim']))

    # Find combinations that are only in duplic (not in mapped)
    only_in_duplic = duplic_combinations - mapped_combinations

    # Find combinations that are only in mapped (not in duplic)
    only_in_mapped = mapped_combinations - duplic_combinations

    # All combinations that don't appear in both
    not_in_both = only_in_duplic.union(only_in_mapped)

    print("Combinations only in duplic:")
    for combo in only_in_duplic:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

    print("\nCombinations only in mapped: "+str(len(only_in_mapped)))
    for combo in only_in_mapped:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

Combinations only in duplic:

Combinations only in mapped: 26
  region_bld: C-WEU-BEL, clim: Zone_5A Cool Humid
  region_bld: C-WEU-DEU, clim: Zone_6A Cold Humid
  region_bld: C-WEU-FRA, clim: Zone_1A Very hot Humid
  region_bld: C-WEU-GRC, clim: Zone_5A Cool Humid
  region_bld: C-WEU-FRA, clim: Zone_2A Hot Humid
  region_bld: C-EEU-SVN, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-IRL, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-GRC, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-PRT, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-SWE, clim: Zone_8A Subarctic/arctic Humid
  region_bld: C-WEU-AUT, clim: Zone_7A Very cold Humid
  region_bld: C-EEU-HRV, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-FRA, clim: Zone_3C Warm Marine
  region_bld: C-WEU-GRC, clim: Zone_5C Cool Marine
  region_bld: C-WEU-FRA, clim: Zone_0A Extremely hot Humid
  region_bld: C-WEU-ITA, clim: Zone_7A Very cold Humid
  region_bld: C-WEU-SWE, clim: Zone_7A Very cold Humid
  region_bld: C-EEU-SVN, clim: Zone_7A V

In [29]:
#Check which files contain clim as index --> most files do
import os
import glob

folder_path = "/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/message-ix-buildings/message_ix_buildings/sturm/data/input_csv_NUTS_2025_resid"#"/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/messageix-buildings-subnational/2025_EU/input_resid"
files_with_clim = [os.path.basename(f) for f in glob.glob(os.path.join(folder_path, "*.csv")) if 'clim' in pd.read_csv(f, nrows=0).columns]
print("Files with 'clim' column:")
for filename in files_with_clim:
    print(f"  {filename}")

Files with 'clim' column:
  stock_baseyear_resid_rev_share_nuts_bld.csv
  bld_shr_access_cool_resid_ssp2_rev.csv
  bld_shr_access_cool_resid_ssp2_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev.csv
  shr_need_heat_resid_rev_nuts_bld.csv
  pop_clim_rev_SSP2_nuts_bld.csv
  shr_need_cool_resid_rev_nuts_bld.csv
  bld_share_mat_resid_ssp2_rev_nuts_bld.csv
  shr_need_heat_resid_rev.csv
  heat_intensity_rev_nuts_bld.csv
  stock_baseyear_resid_rev_share_nuts.csv
  climatic_zones_rev_nuts.csv
  climatic_zones_rev_nuts_bld.csv
  bld_shr_access_cool_resid_ssp2_rev_nuts_bld.csv
  heat_intensity_rev_nuts.csv
  cool_days_nuts_bld.csv
  cool_intensity_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev_nuts.csv
  shr_need_heat_resid_rev_nuts.csv
  shr_need_cool_resid_rev_nuts.csv
  heat_intensity_rev.csv
  pop_clim_rev_SSP2.csv
  cool_days_nuts.csv
  cool_days.csv
  climatic_zones_rev.csv
  shr_need_cool_resid_rev.csv
  stock_baseyear_resid_rev_nuts.csv
  cool_days_rev_nuts.csv
  pop_clim_rev_SSP2_nuts.csv
  hea

In [7]:
# STOCK BASEYEAR SHARES

#Dwelling stock: urbanity, arch, climate zone, income class, year of construction, 

#bld_shr_arch_resid

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOB_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwob_r3_en.csv', usecols=['Housing', 'Type of building','geo', 'OBS_VALUE']) #dwellings by region and type of building
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input = input.reset_index().rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input = input.groupby(by=['region_bld','region_nuts', 'clim', 'arch']).sum()

input = input.div(input.groupby(level=[0,1,2]).sum()) #arch shares of dwellings
input = input.unstack(3).fillna(0.5).stack(future_stack=True) #add missing arch types, and assume equal split

#add average data for missing regions
def add_missing_nuts(df, region_bld, new_nuts):
    climate = region_nuts_clim[region_nuts_clim['region_nuts']==i].clim.unique()[0]
    region_avg = (
        df.groupby(['region_bld', 'clim', 'arch'])
          .mean()
          .loc[(region_bld, climate, slice(None))]
          .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
          .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
          .reorder_levels(['region_bld', 'region_nuts', 'clim', 'arch'])
    )
    return pd.concat([df, region_avg]).sort_index()

for i in set(region_nuts_lab.region_nuts) - set(input.reset_index().region_nuts):
    input = add_missing_nuts(input, i[:9], i)

#dwelling stock national distribution
filename = 'bld_shr_arch_resid'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
input_trend_original = input_trend_original[input_trend_original.region_gea.isin(region_nuts_lab.region_gea.unique())]

#adding years
input_trend_out = pd.concat({year: input for year in input_trend_original.year.unique()},names=['year'])
#adding urban and rural typology
input_trend_out = pd.concat({urt: input_trend_out for urt in input_trend_original.urt.unique()},names=['urt']).reset_index()
#add mat
input_trend_out['mat'] = 'perm'

#align column order with input trend original, and fill missing values (only in 1 NUTSxarch combination)
input_trend_out = input_trend_out.fillna(0).rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1)
input_trend_out = input_trend_out.reindex(columns=['region_bld', 'region_nuts', 'urt', 'mat', 'arch', 'year', 'value'])

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original.copy()
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop(['value', 'region_gea'], axis=1).reset_index(drop=True),\
                                                                    out.drop(['value', 'region_bld'], axis=1).drop('region_nuts', axis=1).drop_duplicates().reset_index(drop=True)))

Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  False


In [8]:
#construction period
#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOP_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwop_r3_en.csv', usecols=['Housing', 'y_const','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
#input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input = input.loc[~input['y_const'].isin(['UNK', 'TOTAL'])] #only direct numbers
input['y_const']=input.y_const.str[-4:]
input.replace({'2016':'2020', '1919':'1945'}, inplace=True)
input = input.groupby(by=['geo', 'y_const', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input.reset_index()
input = input.reset_index().rename({'geo':'NUTS-3 Code', 'y_const':'yr_con'}, axis=1)

#adding missing region labels
input = nuts3code_to_region_nuts(input)
input = input.set_index(['region_bld','region_nuts', 'yr_con']).div(input.groupby(by=['region_bld','region_nuts']).sum().drop('yr_con', axis=1)).reset_index() #TODO: - interpolate input_shares to more detailed years

input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories

input.set_index(['region_bld','region_nuts', 'clim', 'yr_con'], inplace=True)

#add average data for missing regions
def add_missing_nuts(df, region_bld, new_nuts):
    climate = region_nuts_clim[region_nuts_clim['region_nuts']==i].clim.unique()[0]
    if len(df.reset_index()[(df.reset_index()['region_bld']==region_bld) & (df.reset_index()['clim']==climate)])>0:
        region_avg = (
            df.groupby(['region_bld', 'clim', 'yr_con'])
            .mean(numeric_only=True)
            .loc[(region_bld, climate, slice(None))]
            .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
            .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
            .reorder_levels(['region_bld', 'region_nuts', 'clim', 'yr_con'])
        )
    else:
        region_avg = (
            df.groupby(['region_bld', 'yr_con'])
            .mean(numeric_only=True)
            .loc[(region_bld, slice(None))]
            .assign(region_bld=region_bld, region_nuts=new_nuts, clim=climate)
            .set_index(['region_bld', 'region_nuts', 'clim'], append=True)
            .reorder_levels(['region_bld', 'region_nuts', 'clim', 'yr_con'])
        )        
    return pd.concat([df, region_avg]).sort_index()

for i in set(region_nuts_lab.region_nuts) - set(input.reset_index().region_nuts):
    input = add_missing_nuts(input, i[:9], i)


#dwelling stock national distribution
filename = 'stock_baseyear_resid_rev_share'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

input_trend_original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]

#adding original columns
#adding arch
input_trend_out = pd.concat({arch: input for arch in input_trend_original.arch.unique()},names=['arch'])
#adding urban and rural typology
input_trend_out = pd.concat({urt: input_trend_out for urt in input_trend_original.urt.unique()},names=['urt']).reset_index()
#adding bld age categories
replacement_map = dict(zip(input_trend_original[['yr_con', 'bld_age']].drop_duplicates().yr_con.astype(str), input_trend_original[['yr_con', 'bld_age']].drop_duplicates().bld_age))
input_trend_out['bld_age'] = input_trend_out.yr_con.map(replacement_map)
#add mat
input_trend_out['mat'] = 'perm'

input_trend_out = input_trend_out.rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1)

#input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out['region_gea'] = input_trend_out.region_bld.str[2:5]
input_trend_out['yr_con'] = input_trend_out['yr_con'].astype(int)
input_trend_out = input_trend_out.reindex(columns=['region_bld', 'region_gea', 'region_nuts', 'urt', 'clim', 'mat', 'arch', 'yr_con', 'bld_age', 'value'])
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original.copy()
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1)[original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), \
                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))
#not all contained because of differing construction year

Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  False


In [ ]:
#NOTE: values for shr_need and bld_mat are only binary (0,1) in Europe in original files
def detail_other_inputs(idx_full):
    # Non-specified Inputs Detailed: expanding tables to cover all NUTS regions
    column_list = ['region_nuts']
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP\xa02.00'].to_list()
    replacement_map = {#inc_cl
                    'q1':1,
                    'q2':2,
                    'q3':3,
                    #bld_age
                    'ns':0,
                    'p1':1,
                    'p2':2,
                    'p3':3,
                    #eneff
                    'ns':0, 
                    's1':1, 
                    's2':2, 
                    's3':3, 
                    'sr11_std':4,
                    'sr21_std':5, 
                    'sr31_std':6, 
                    's51_std':7, 
                    'sr12_low':8,
                    'sr22_low':9,  
                    'sr32_low':10,
                    's52_low':11, 
                    #clim
                    'Zone_0A Extremely hot Humid':0, 
                    'Zone_1A Very hot Humid':1,
                    'Zone_2A Hot Humid':2, 
                    'Zone_2B Hot Dry':2.5, 
                    'Zone_3A Warm Humid':3,
                    'Zone_3B Warm Dry':3.3, 
                    'Zone_3C Warm Marine':3.7, 
                    'Zone_4A Mixed Humid':4,
                    'Zone_4C Mixed Marine':4.5,
                    'Zone_5A Cool Humid':5,
                    'Zone_5C Cool Marine':5.5, 
                    'Zone_6A Cold Humid':6,
                    'Zone_7A Very cold Humid':7, 
                    'Zone_8A Subarctic/arctic Humid':8,
                    #arch
                    'mfh':0,
                    'sfh':1,
                    'inf':2,
                    np.inf:2,
                    #mat
                    'sub':0,
                    'perm':1,
                    #region_gea
                    'WEU':0,
                    'EEU':1,
                    #urt
                    'rur':0,
                    'urb':1,
                    }
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share', 'stock_baseyear_resid_rev', 'pop_clim_rev_SSP2', 'hh_size_rev', 'floor_resid_ssp2_rev']: #for filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days_rev', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev', 'bld_shr_access_cool_resid_ssp2_rev', 'bld_share_mat_resid_ssp2_rev']:
            input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
            if ('region_bld' in input.columns) and ('clim' in input.columns):   #there are two more files which have bld but not clim, households and floor space, those are disaggregated below
                input_cols = list(input.columns.drop('value'))
                print(input_cols)
                #merge files
                idx_merge = idx_full[input_cols + ['region_nuts']].drop_duplicates()
                input_nuts = pd.merge(input, idx_merge, on=input_cols, how='right')
                input_nuts_cols = list(input_nuts.columns.drop('value'))
                df = input_nuts.set_index(input_nuts_cols)

                #numerical encoding of ordinal, binary and categorical data
                input_nuts = input_nuts.replace(replacement_map)
                input_nuts = pd.concat([input_nuts, pd.get_dummies(input_nuts.region_bld, dtype=float)], axis=1)
                input_nuts = input_nuts.drop(['region_bld', 'region_nuts'], axis=1)

                #impute values for missing country-climate zone combinations based on nearest neighbours
                input_nuts = pd.DataFrame(KNNImputer(missing_values=np.nan, n_neighbors=3, weights='uniform')\
                            .fit(input_nuts).transform(input_nuts), index=df.index, columns=input_nuts.columns)['value'].reset_index()

                input_nuts['value'] = round(input_nuts.value, 8)

                #collect column labels
                for i in input_nuts_cols:
                    if i not in column_list:
                        column_list.append(i)
                
                #export
                input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
                input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

                #checks
                print(input_nuts_cols)
                original = input[input.region_bld.isin(region_nuts_lab.region_bld.unique())]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

                
                print(filename + ' DONE')
    print(column_list)
    
    return column_list


idx_full = pd.read_csv('/Volumes/KINGSTON/data/idx_full.csv')
detail_other_inputs(idx_full)

['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/3698286448.py:71: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'region_nuts']
Missing values:  0
Null before:  5508
Null after:  125820
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
bld_share_mat_resid_ssp2_rev DONE
['region_bld', 'clim', 'urt', 'inc_cl', 'year']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/3698286448.py:71: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'urt', 'inc_cl', 'year', 'region_nuts']
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
bld_shr_access_cool_resid_ssp2_rev DONE
['region_bld', 'clim', 'urt', 'arch', 'eneff']


...

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/3698286448.py:71: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'urt', 'arch', 'eneff', 'region_nuts']
Missing values:  0
Null before:  84
Null after:  1040
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
shr_need_cool_resid_rev DONE
['region_bld', 'clim', 'urt', 'arch', 'eneff']


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_21271/3698286448.py:71: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input_nuts = input_nuts.replace(replacement_map)


['region_bld', 'clim', 'urt', 'arch', 'eneff', 'region_nuts']
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
shr_need_heat_resid_rev DONE
['region_nuts', 'region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'arch', 'eneff']


['region_nuts',
 'region_bld',
 'clim',
 'urt',
 'inc_cl',
 'mat',
 'year',
 'arch',
 'eneff']

In [68]:
filename = 'bld_share_mat_resid_ssp2_rev'
input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')

In [69]:
input

,region_bld,clim,urt,inc_cl,mat,year,region_nuts,value
0,C-WEU-AUT,Zone_5A Cool Humid,rur,q1,sub,2015,C-WEU-AUT-AT111,0.00
1,C-WEU-AUT,Zone_5A Cool Humid,urb,q1,sub,2015,C-WEU-AUT-AT111,0.00
2,C-WEU-AUT,Zone_4A Mixed Humid,rur,q1,sub,2015,C-WEU-AUT-AT112,0.00
3,C-WEU-AUT,Zone_4A Mixed Humid,urb,q1,sub,2015,C-WEU-AUT-AT112,0.00
4,C-WEU-AUT,Zone_5A Cool Humid,rur,q1,sub,2015,C-WEU-AUT-AT113,0.00
...,...,...,...,...,...,...,...,...
251635,C-EEU-SVK,Zone_5A Cool Humid,urb,q3,perm,2100,C-EEU-SVK-SK032,1.00
251636,C-EEU-SVK,Zone_6A Cold Humid,rur,q3,perm,2100,C-EEU-SVK-SK041,1.00
251637,C-EEU-SVK,Zone_6A Cold Humid,urb,q3,perm,2100,C-EEU-SVK-SK041,1.00
251638,C-EEU-SVK,Zone_5A Cool Humid,rur,q3,perm,2100,C-EEU-SVK-SK042,1.00


In [ ]:
                # Heat Operation Hours (only input with region_gea + clim differentiation)

                filename = 'heat_operation_hours_ssp2'
                input = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

                region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input.year.unique()), keys=list(input.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
                region_nuts_clim_year['region_gea'] = region_nuts_clim_year.region_bld.str[2:5]
                region_nuts_clim_year.drop(['urt'], axis=1, inplace=True)
                region_nuts_clim_year.drop_duplicates(inplace=True)

                input_nuts = pd.merge(input, region_nuts_clim_year, how='right')


                input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

                #checks
                original = input[input.region_gea.isin(['WEU', 'EEU'])]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_gea').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop(['region_nuts', 'region_bld'], axis=1).drop_duplicates().sort_values('region_gea').reset_index(drop=True)))

                print(filename + ' DONE')

Missing values:  0
Null before:  216
Null after:  72
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  False
heat_operation_hours_ssp2 DONE


In [8]:
# Household size: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNO_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input.replace({'GE11':'12'}, inplace=True) #aligning labels with model
input = input.loc[~input['n_person'].isin(['3-5', '6-10', 'GE6', 'TOTAL'])] #only direct numbers
input['population'] = input['n_person'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum(numeric_only=True)
input['hh_size'] = input['population'] / input['OBS_VALUE'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings
input = input.reset_index().drop(['OBS_VALUE'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)

input = input.groupby(by=['region_bld','region_nuts', 'arch']).agg({'population':'sum', 'hh_size':'mean'})

input_hh = input.copy()

# household size: trend
filename = 'hh_size_rev'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')
input_trend_original = pd.concat([input_trend_original.set_index(['region_bld', 'urt', 'year']), input_trend_original.set_index(['region_bld', 'urt', 'year'])], keys=['mfh', 'sfh'], names=['arch','region_bld', 'urt', 'year']).reorder_levels([1,2,0,3])

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original.reset_index(), region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.nan).rename({'hh_size':'value'}, axis=1).drop('population', axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'urt','arch']).mean().drop('year', axis=1)) #QQ: why is there only hhd size trend in Croatia and no other EU countries?
#input_trend_diff = input_trend_diff.assign(value=1) #replacing Croatian trend in household sizes with stagnant household sizes
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original.reset_index()[input_trend_original.reset_index().region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_9310/459963305.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [9]:
# Floor area: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNR_R3
input = pd.read_csv('data/input_detailing_NUTS/estat_cens_21dwbnr_r3_filtered_en.csv', usecols=['area', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['area'].isin(['UNK'])] #only direct numbers
input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model
input['fa_total'] = input['area'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input = input.reset_index().drop(['OBS_VALUE', 'area'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)
input = nuts3code_to_region_nuts(input)
input = input.groupby(by=['region_bld','region_nuts', 'arch']).sum()
input['fa_pc'] = input['fa_total'] / input_hh['population'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings

#fill up missing values in input['fa_pc']
# floor area: trend and missing values
filename = 'floor_resid_ssp2_rev'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv')

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.nan).rename({'fa_pc':'value'}, axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'mat','arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean(numeric_only=True).drop('year', axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_9310/242159404.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [16]:
# Population: current
#import EUROSTAT projections: https://doi.org/10.2908/PROJ_19RP3
input = pd.read_csv('data/input_detailing_NUTS/estat_proj_19rp3_filtered_en.csv', usecols=['geo', 'TIME_PERIOD','OBS_VALUE']) #residents per territory and year
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.rename({'geo':'NUTS-3 Code'}, axis=1)
input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input.rename({'TIME_PERIOD':'year', 'OBS_VALUE':'value'}, axis=1, inplace=True)
input = input[input['year'].isin(list(range(2020,2101,5)))]
input = (input.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum()/1e6)
#input = (input.set_index(['region_bld', 'region_nuts', 'clim', 'year'])/1e6)#.reset_index() #million resident stock per year

#population national distribution
filename = 'pop_clim_rev_SSP2'
input_trend_original = pd.read_csv('data/input_csv_SSP_2023_resid/'+filename+'.csv') #million residents per country, urbanity, climate zone, and year

#merge index by duplicating values in original model data
region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input_trend_original.year.unique()), keys=list(input_trend_original.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
input_trend = pd.merge(input_trend_original, region_nuts_clim_year, how='right') #duplicating population across same urt, clim, country
input_trend = input_trend.set_index(['region_bld', 'region_nuts', 'urt', 'clim', 'year'])

#fill those regions for which no matching country-climate combinations were available, using simply the matching country-urbanity combination
input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4]).mean() #country, urbanity, year used for filling nan values
filler_mapped = input_trend_mean.reindex(input_trend.index.droplevel([1, 3])) #nuts and climate not used
filler_mapped.index = input_trend.index
input_trend = input_trend.fillna(filler_mapped).reset_index()

#normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination
nuts_in_reg = region_nuts_clim.drop('urt', axis=1).drop_duplicates()[['region_bld', 'clim']].value_counts().reset_index() #number of nuts within each country-climate combination
nuts_in_reg.columns = ['region_bld', 'clim', 'value']
input_trend = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(nuts_in_reg.set_index(['region_bld', 'clim'])).reset_index() #normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination

#include input values fom EUROSTAT
input_trend_regurtarch = input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum(numeric_only=True) #population per nuts and year
input_trend_regurtarch.update(input.replace(0,np.nan)) #

#expand pop by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum(numeric_only=True))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_9310/2092145438.py:23: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4]).mean() #country, urbanity, year used for filling nan values


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


## Increase resolution in aggregate level from world regions --> national

In [5]:
    #Aggregating NUTS3 files back to NUTS0='region_bld'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP2-NUTS'].dropna().to_list()
    input_list = [i.removesuffix('_nuts') for i in input_list]
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')
            print(filename, input.columns)

hh_size_rev Index(['region_bld', 'region_nuts', 'urt', 'arch', 'year', 'value'], dtype='object')
floor_resid_ssp2_rev Index(['region_bld', 'region_nuts', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
bld_shr_arch_resid Index(['region_bld', 'region_nuts', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
bld_share_mat_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'region_nuts',
       'value'],
      dtype='object')
bld_shr_access_cool_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'year', 'region_nuts', 'value'], dtype='object')
heat_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'region_nuts', 'value'], dtype='object')
cool_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'region_nuts', 'value'], dtype='object')
cool_days Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'region_nuts', 'value'], dtype='object')
shr_need_cool_resid_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff'

In [25]:
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP\xa02.00'].dropna().to_list()
    input_list = [i.removesuffix('_nuts') for i in input_list]
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'.csv')
            print(filename, input.columns)

ct_bld_age Index(['bld_age_id', 'bld_age_name', 'mat', 'year_i', 'year_f'], dtype='object')
ct_bld Index(['region_gea', 'urt', 'arch', 'mat'], dtype='object')
ct_eneff Index(['eneff', 'mat', 'bld_age'], dtype='object')
ct_fuel_comb Index(['mat', 'fuel_heat', 'fuel_cool', 'mod_decision'], dtype='object')
ct_fuel_dhw Index(['mat', 'fuel_heat', 'fuel_cool', 'res_solar', 'mod_decision'], dtype='object')
ct_ren_eneff Index(['mat', 'eneff_i', 'eneff_f'], dtype='object')
ct_inc_cl Index(['urt', 'inc_cl'], dtype='object')
ct_tenr Index(['mat', 'tenr'], dtype='object')
hh_size_rev Index(['region_bld', 'urt', 'year', 'value'], dtype='object')
floor_resid_ssp2_rev Index(['region_bld', 'urt', 'arch', 'mat', 'year', 'value'], dtype='object')
shr_hh_tenr Index(['region_gea', 'urt', 'mat', 'tenr', 'year', 'value'], dtype='object')
bld_shr_arch_resid Index(['region_gea', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
bld_share_mat_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', '

In [12]:
    #Aggregating NUTS3 files back to NUTS0='region_bld'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP2-NUTS-BLD'].dropna().to_list()
    input_list = [i.removesuffix('_nuts_bld') for i in input_list]
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts_bld.csv')
            print(filename, input.columns)

hh_size_rev Index(['region_bld', 'urt', 'arch', 'year', 'value'], dtype='object')
floor_resid_ssp2_rev Index(['region_bld', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
bld_shr_arch_resid Index(['region_bld', 'urt', 'mat', 'arch', 'year', 'value'], dtype='object')
bld_share_mat_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'value'], dtype='object')
bld_shr_access_cool_resid_ssp2_rev Index(['region_bld', 'clim', 'urt', 'inc_cl', 'year', 'value'], dtype='object')
heat_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
cool_intensity_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
cool_days Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
shr_need_cool_resid_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
shr_need_heat_resid_rev Index(['region_bld', 'clim', 'urt', 'arch', 'eneff', 'value'], dtype='object')
st

input_list
- 'regions_R61', #no change necessary
- 'climatic_zones_rev' #no change necessary
- 'pop_clim_rev_SSP2' #no change necessary

- 'hh_size_rev' #mean weighted by number of dwellings (= pop / hhsize * arch_share) in each cell with arch weighted by bld_arch_share; by 'urt', 'arch', 'year'
 'floor_resid_ssp2_rev', #mean weighted by number of dwellings (=pop/hhsize*arch_share), ignore mat, always same; 'urt', 'mat', 'arch', 'year',
 'bld_shr_arch_resid', #mean weighted by number of dwellings (=pop/hhsize*arch_share), ignore mat, always same; 'urt', 'mat', 'arch', 'year',
 'stock_baseyear_resid_rev_share' #mean weighted by number of dwellings

- 'bld_share_mat_resid_ssp2_rev', #use original files, since no primary data injected
 'bld_shr_access_cool_resid_ssp2_rev', #use original files, since no primary data injected
 'heat_intensity_rev', #use original files, since no primary data injected
 'cool_intensity_rev', #use original files, since no primary data injected
 'cool_days', #use original files, since no primary data injected
 'shr_need_cool_resid_rev', #use original files, since no primary data injected
 'shr_need_heat_resid_rev', #use original files, since no primary data injected


In [18]:
    #Aggregating NUTS3 files back to NUTS0='region_bld'
    input_list = pd.read_csv('data/input_list_resid_2025_11_06_nuts.csv')['SSP2-NUTS'].dropna().to_list()
    input_list = [i.removesuffix('_nuts') for i in input_list]

    #calculating dwelling weights of dwellings in NUTS per country
    pop = pd.read_csv('data/input_csv_NUTS_2025_resid/pop_clim_rev_SSP2_nuts.csv')
    pop['mat'] = 'perm'
    hhsize = pd.read_csv('data/input_csv_NUTS_2025_resid/hh_size_rev_nuts.csv')
    hhsize['mat'] = 'perm'
    hhsize = hhsize.set_index(keys=hhsize.columns.drop('value').to_list())
    arch_shr = pd.read_csv('data/input_csv_NUTS_2025_resid/bld_shr_arch_resid_nuts.csv')
    arch_shr = arch_shr.set_index(keys=arch_shr.columns.drop('value').to_list())

    dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
    dw_weights = dwellings.div(dwellings.groupby(level=[0,2,3,4]).sum())

    for filename in input_list:
        if filename in ['hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            if filename == 'stock_baseyear_resid_rev_share':
                dw_weights = dw_weights.xs(2020, level='year')
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts_bld.csv', index=False)

            if filename == 'stock_baseyear_resid_rev_share':
                dw_weights = dwellings.div(dwellings.groupby(level=[0,2,3,4]).sum())

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename not in ['regions_R61','climatic_zones_rev', 'pop_clim_rev_SSP2', 'hh_size_rev','floor_resid_ssp2_rev', 'bld_shr_arch_resid', 'stock_baseyear_resid_rev_share']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')
            
            #mean weighted by dwellings per NUTS, rurality and year
            if filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev']:
                dw_weights_ = dw_weights.xs(2020, level='year')
            else:
                dw_weights_ = dw_weights.copy()
            input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)
            input_bld = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'.csv')
            input_bld = input_bld.set_index(input_bld.columns.drop('value').to_list())
            input_nuts.update(input_bld)
            input_nuts = input_nuts.reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['pop_clim_rev_SSP2']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['regions_R61','climatic_zones_rev']:
            input = pd.read_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts.csv')
            
            input_nuts = input.drop('region_nuts', axis=1).drop_duplicates()
            input_nuts.to_csv('data/input_csv_NUTS_2025_resid/'+filename+'_nuts_bld.csv', index=False)
                            
            print(filename + ' DONE')

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dwellings = pop.groupby(by=pop.columns.drop(['clim','value']).to_list()).sum(numeric_only=True).mul(1e6).div((hhsize*arch_shr).groupby(level=[0,1,2,4,5]).sum()).fillna(method='bfill')
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:66: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


regions_R61 DONE
climatic_zones_rev DONE
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
pop_clim_rev_SSP2 DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:24: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
hh_size_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:24: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()


...

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:47: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)


Missing values:  0
Null before:  1040
Null after:  162
Duplicates:  0
shr_need_cool_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:47: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights_).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True)


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
shr_need_heat_resid_rev DONE
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
stock_baseyear_resid_rev_share DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_24735/544265733.py:24: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  input_nuts = (input.set_index(input.columns.drop('value').to_list()) * dw_weights).reset_index().groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum(numeric_only=True).reset_index()
